In [5]:
import numpy as np
import geopandas as gpd
import os
import re

def analyze_ml_results(grid_file, npy_folder):
    """Analyse les résultats ML et calcule les résolutions"""
    
    # Charger la grille
    grid = gpd.read_file(grid_file)
    npy_files = [f for f in os.listdir(npy_folder) if f.endswith('.npy')]
    
    print(f"Grille: {len(grid)} tuiles")
    print(f"Fichiers .npy: {len(npy_files)}")
    
    for npy_file in npy_files:
        print(f"\n{'='*50}")
        print(f"Fichier: {npy_file}")
        
        # Extraire l'ID du nom de fichier
        tile_id = extract_tile_id_corrected(npy_file)
        print(f"ID de tuile extrait: {tile_id}")
        
        # Charger le dictionnaire
        try:
            data_dict = np.load(os.path.join(npy_folder, npy_file), allow_pickle=True).item()
            print(f"Type: {type(data_dict)}")
            print(f"Clés du dictionnaire: {list(data_dict.keys())}")
            
            # Analyser le contenu de chaque clé
            for key, value in data_dict.items():
                print(f"\n  Clé '{key}':")
                print(f"    Type: {type(value)}")
                if hasattr(value, 'shape'):
                    print(f"    Shape: {value.shape}")
                    print(f"    Dtype: {value.dtype}")
                    
                    # Si c'est une image 128x128, calculer la résolution
                    if len(value.shape) >= 2 and value.shape[-2:] == (128, 128):
                        print(f"    ✓ Image 128x128 détectée!")
                        
                        # Trouver la tuile correspondante dans la grille
                        if tile_id is not None and tile_id < len(grid):
                            tile_row = grid[grid['id'] == tile_id]
                            if len(tile_row) > 0:
                                bounds = tile_row.iloc[0].geometry.bounds
                                width_m = bounds[2] - bounds[0]
                                height_m = bounds[3] - bounds[1]
                                
                                resolution_x = width_m / 128
                                resolution_y = height_m / 128
                                
                                print(f"    Dimensions géographiques: {width_m:.1f}m x {height_m:.1f}m")
                                print(f"    Résolution: {resolution_x:.2f} x {resolution_y:.2f} m/pixel")
                            else:
                                print(f"    ⚠ Tuile ID {tile_id} non trouvée dans la grille")
                        else:
                            print(f"    ⚠ ID de tuile invalide: {tile_id}")
                    else:
                        if hasattr(value, 'shape'):
                            print(f"    Dimensions: {value.shape}")
                elif isinstance(value, (int, float, str)):
                    print(f"    Valeur: {value}")
                else:
                    print(f"    Contenu: {str(value)[:100]}...")
            
        except Exception as e:
            print(f"Erreur lors du chargement: {e}")

def extract_tile_id_corrected(filename):
    """Extrait correctement l'ID de la tuile du nom de fichier"""
    # Pour vos fichiers: "123_best_fold_5.npy"
    patterns = [
        r'^(\d+)_best_fold_\d+\.npy$',    # 123_best_fold_5.npy
        r'^(\d+)_.*\.npy$',               # 123_quelquechose.npy
        r'^(\d+)\.npy$',                  # 123.npy
    ]
    
    for pattern in patterns:
        match = re.search(pattern, filename)
        if match:
            return int(match.group(1))
    
    print(f"⚠ Impossible d'extraire l'ID de: {filename}")
    return None

# Fonction pour examiner un fichier spécifique en détail
def examine_specific_file(filepath):
    """Examine un fichier .npy spécifique en détail"""
    print(f"Examen détaillé de: {filepath}")
    
    try:
        data = np.load(filepath, allow_pickle=True).item()
        
        print(f"Type: {type(data)}")
        if isinstance(data, dict):
            print(f"Nombre de clés: {len(data)}")
            
            for key, value in data.items():
                print(f"\n=== Clé: {key} ===")
                print(f"Type de valeur: {type(value)}")
                
                if hasattr(value, 'shape'):
                    print(f"Shape: {value.shape}")
                    print(f"Dtype: {value.dtype}")
                    
                    if hasattr(value, 'min') and hasattr(value, 'max'):
                        try:
                            print(f"Min/Max: {value.min():.3f} / {value.max():.3f}")
                        except:
                            pass
                    
                    # Si c'est une image, montrer quelques statistiques
                    if len(value.shape) >= 2:
                        print(f"Forme: {value.shape}")
                        if len(value.shape) == 3:
                            print(f"Nombre de canaux: {value.shape[0] if value.shape[0] < value.shape[1] else value.shape[2]}")
                        
                else:
                    print(f"Valeur: {value}")
                    
    except Exception as e:
        print(f"Erreur: {e}")

# Usage
print("=== ANALYSE COMPLÈTE ===")
analyze_ml_results("grid_indre_loire_128_lambert93.geojson", "test_tuiles/")

print("\n=== EXAMEN DÉTAILLÉ D'UN FICHIER ===")
examine_specific_file("test_tuiles/0_best_fold_5.npy")

=== ANALYSE COMPLÈTE ===
Grille: 6630 tuiles
Fichiers .npy: 5

Fichier: 100_best_fold_5.npy
ID de tuile extrait: 100
Type: <class 'dict'>
Clés du dictionnaire: ['center_mask', 'saliency', 'heatmap', 'semantic', 'size', 'confidence', 'centerness', 'instance_masks', 'instance_boxes', 'pano_instance', 'pano_semantic', 'best_fold', 'all_confidences', 'tile_id']

  Clé 'center_mask':
    Type: <class 'numpy.ndarray'>
    Shape: (1, 128, 128)
    Dtype: bool
    ✓ Image 128x128 détectée!
    Dimensions géographiques: 1280.0m x 1280.0m
    Résolution: 10.00 x 10.00 m/pixel

  Clé 'saliency':
    Type: <class 'numpy.ndarray'>
    Shape: (1, 1, 128, 128)
    Dtype: float32
    ✓ Image 128x128 détectée!
    Dimensions géographiques: 1280.0m x 1280.0m
    Résolution: 10.00 x 10.00 m/pixel

  Clé 'heatmap':
    Type: <class 'numpy.ndarray'>
    Shape: (1, 1, 128, 128)
    Dtype: float32
    ✓ Image 128x128 détectée!
    Dimensions géographiques: 1280.0m x 1280.0m
    Résolution: 10.00 x 10.00 m/pi